# FastGS — Synthetic Specular Dataset Pipeline

> **Kernel**: `thesis_env` (Set via Kernel → Change Kernel after running `bosch_setup_thesis.ipynb` once to register it)

Runs the full FastGS training sweep on the 8 Synthetic Specular scenes inside the BOSCH server environment, reading datasets directly from `/home/ghp4hc/datasets/datasets/synthetic_specular`.

**What this notebook does:**
1. **Proxy & Env Check**: Sets up environment variables for the BOSCH server.
2. **Dataset Verification**: Verifies the presence of the Synthetic Specular dataset root `/home/ghp4hc/datasets/datasets/synthetic_specular`.
3. **Dataset Validation**: Verifies `transforms_train.json` or `transforms.json` exists for all 8 scenes (`ashtray`, `dishes`, `headphone`, `jupyter`, `lock`, `plane`, `record`, `teapot`).
4. **Run Sweep**: Invokes the `run_synthetic_specular.sh` batch script directly in `FastGS` with `DATA_ROOT="/home/ghp4hc/datasets/datasets/synthetic_specular"`, logging outputs to `synthetic_specular_fastgs_run.log`.
5. **Results Aggregation**: Formats and prints quantitative metrics (`results.json`) and final Gaussian point counts in a neat table.
6. **Output Archiving**: Zips the results to `fastgs_output_synthetic_specular.zip` in the parent directory.
7. **Hugging Face Upload & Auto-Cleanup**: Automatically uploads the zipped output results back to `DiBiay/fastgs-synthetic-specular-result`, then purges local zip & output folder to save disk space.

## c00 — Proxy Settings
Sets the BOSCH proxy for external connectivity.

In [ ]:
# ── Proxy (required for HF / huggingface cache / diagnostic endpoints) ────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

## c01 — Config & Kernel Check
Defines paths and double-checks if the correct virtual environment kernel is loaded.

In [ ]:
# ── Configurations & environment variables check ─────────────────────────────
import os
import sys

HOME = os.path.expanduser('~')

# Robustly resolve FastGS repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/FastGS'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/FastGS'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'FastGS')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'FastGS')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'FastGS')

ENV_NAME = 'thesis_env'

print(f'Active Python      : {sys.executable}')
print(f'Active Kernel name : {ENV_NAME}')
print(f'Repository Root    : {REPO_ROOT}')

assert REPO_ROOT in sys.executable or ENV_NAME in sys.executable or '.conda' in sys.executable, \
    f"WARNING: You are not running on the '{ENV_NAME}' kernel! Please select Kernel -> Change Kernel -> Python ({ENV_NAME})"

## c02 — Imports & GPU Validation
Verifies hardware detection and compiles custom modules availability.

In [ ]:
# ── Verification of PyTorch & custom submodules ────────────────────────────────
import torch
print('PyTorch version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name :', torch.cuda.get_device_name(0))
    print('Compute Cap.    :', torch.cuda.get_device_capability(0))

import diff_gaussian_rasterization
import simple_knn
import fused_ssim
print('rasterizer      : OK')
print('simple-knn      : OK')
print('fused-ssim      : OK')

try:
    import huggingface_hub
    print('huggingface_hub : OK')
except ImportError:
    print('huggingface_hub : MISSING (will auto-install during the upload step)')

## c03 — Verify Dataset Path
Checks that the Synthetic Specular source dataset is available on the server.

In [ ]:
# ── Verify Dataset Directory ──────────────────────────────────────────────────
import os
import sys

src_root = "/home/ghp4hc/datasets/datasets/synthetic_specular"
assert os.path.exists(src_root), f"Dataset path not found at {src_root}! Check that the dataset is downloaded."
print(f"✅ Found dataset source root: {src_root}")

print(f"\n📂 Source datasets directory content:")
print(os.listdir(src_root))

## c04 — Verify Synthetic Specular Dataset Layout
Validates all 8 scenes and checks for `transforms_train.json` or `transforms.json`.

In [ ]:
# ── Verify Synthetic Specular Layout (all 8 scenes) ───────────────────────────
import os

src_root = "/home/ghp4hc/datasets/datasets/synthetic_specular"
SYNTHETIC_SCENES = [
    "ashtray", "dishes", "headphone", "jupyter",
    "lock", "plane", "record", "teapot"
]

print(f"Verifying layouts directly under {src_root} ...")
missing = []
for scene in SYNTHETIC_SCENES:
    scene_dir = os.path.join(src_root, scene)
    if not os.path.isdir(scene_dir):
        status = "MISSING (scene folder not found)"
        missing.append(scene)
    else:
        tf_train = os.path.join(scene_dir, "transforms_train.json")
        tf_single = os.path.join(scene_dir, "transforms.json")
        if os.path.isfile(tf_train):
            status = "OK (transforms_train.json found)"
        elif os.path.isfile(tf_single):
            status = "OK (transforms.json found)"
        else:
            status = f"MISSING (neither transforms_train.json nor transforms.json found; has: {sorted(os.listdir(scene_dir))[:6]})";
            missing.append(scene)
            
    print(f"  {scene:<12s} {status}")

print()
if missing:
    print(f"⚠️  {len(missing)}/{len(SYNTHETIC_SCENES)} scene(s) missing: {missing}")
    print("    run_synthetic_specular.sh will skip these scenes during the sweep.")
else:
    print(f"✅ All {len(SYNTHETIC_SCENES)} scenes are verified and available. Ready for training!")

## c05 — Run FastGS Synthetic Specular Sweep (`run_synthetic_specular.sh`)
Launches the training sweep using `thesis_env` and system CUDA paths.

In [ ]:
# ── Prepare script inputs ──────────────────────────────────────────────────────
import subprocess
import os
import sys

# Robustly resolve FastGS repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/FastGS'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/FastGS'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'FastGS')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'FastGS')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'FastGS')

LOGFILE = os.path.join(os.path.dirname(REPO_ROOT), "synthetic_specular_fastgs_run.log")
DATA_ROOT = "/home/ghp4hc/datasets/datasets/synthetic_specular"

venv_bin = os.path.dirname(sys.executable)
print(f"Virtual environment bin path: {venv_bin}")

cuda_home = os.environ.get('CUDA_HOME', '')
if not cuda_home:
    # Auto-load search script
    search_script = 'which nvcc 2>/dev/null || (for init in /etc/profile /etc/profile.d/modules.sh; do [ -f "$init" ] && source "$init"; done && for mod in cuda/11.7 cuda/11.8 cuda/12.1 cuda/12.6; do module load "$mod" 2>/dev/null; done && which nvcc 2>/dev/null)'
    r_nvcc = subprocess.run(['bash', '-c', search_script], capture_output=True, text=True)
    if r_nvcc.returncode == 0 and r_nvcc.stdout.strip():
        cuda_home = os.path.dirname(os.path.dirname(r_nvcc.stdout.strip()))
    else:
        cuda_home = '/usr/local/cuda'

print(f"CUDA_HOME path: {cuda_home}")

In [ ]:
%%bash -s "$venv_bin" "$cuda_home" "$LOGFILE" "$REPO_ROOT" "$DATA_ROOT"
ENV_BIN=$1
CUDA_HOME=$2
LOGFILE=$3
REPO_ROOT=$4
export DATA_ROOT=$5

export PATH=$ENV_BIN:$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0

cd "$REPO_ROOT"

echo "Running run_synthetic_specular.sh for FastGS (all 8 scenes directly from ${DATA_ROOT}) ..."
bash run_synthetic_specular.sh > "$LOGFILE" 2>&1
STATUS=$?

echo "--- tail of ${LOGFILE} ---"
tail -n 100 "$LOGFILE"
echo "run_synthetic_specular.sh exit status: $STATUS"
if [ $STATUS -ne 0 ]; then
    echo "⚠️  run_synthetic_specular.sh stopped early (STOP_ON_ERROR=True) -- check logs above or at ${LOGFILE}"
fi

## c06 — Quantitative Results Summary
Parses `results.json` and point cloud data from the output directory to print a formatted metrics table.

In [ ]:
# ── Quantitative Results Summary ──────────────────────────────────────────────
import json
import os

# Robustly resolve FastGS repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/FastGS'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/FastGS'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'FastGS')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'FastGS')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'FastGS')

OUTPUT_ROOT = os.path.join(REPO_ROOT, "output", "synthetic_specular")

SYNTHETIC_SCENES = [
    "ashtray", "dishes", "headphone", "jupyter",
    "lock", "plane", "record", "teapot"
]

def fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "-"

def count_gaussians(scene_dir):
    ply_path = os.path.join(scene_dir, "point_cloud", "iteration_30000", "point_cloud.ply")
    if os.path.exists(ply_path):
        try:
            with open(ply_path, 'rb') as f:
                for line in f:
                    line_str = line.decode('ascii', errors='ignore')
                    if line_str.startswith("element vertex"):
                        return int(line_str.split()[2])
        except Exception:
            pass
    return "-"

header = f"{'scene':<12s}{'PSNR':>10s}{'SSIM':>10s}{'LPIPS':>10s}{'#Gaussians':>14s}"
print(header)
print("-" * len(header))

psnr_list, ssim_list, lpips_list = [], [], []

for scene in SYNTHETIC_SCENES:
    out_dir = os.path.join(OUTPUT_ROOT, scene)
    results_path = os.path.join(out_dir, "results.json")

    if not os.path.exists(results_path):
        print(f"{scene:<12s}  (no results.json -- skipped, or sweep did not reach this scene)")
        continue

    with open(results_path) as f:
        results = json.load(f)
    
    iter_key = next(iter(results.keys())) if results else None
    metrics = results.get(iter_key, {}) if iter_key else {}
    
    psnr_val = metrics.get("PSNR")
    ssim_val = metrics.get("SSIM")
    lpips_val = metrics.get("LPIPS")
    n_gauss = count_gaussians(out_dir)
    
    if isinstance(psnr_val, (int, float)):
        psnr_list.append(psnr_val)
    if isinstance(ssim_val, (int, float)):
        ssim_list.append(ssim_val)
    if isinstance(lpips_val, (int, float)):
        lpips_list.append(lpips_val)

    print(f"{scene:<12s}{fmt(psnr_val):>10s}{fmt(ssim_val):>10s}{fmt(lpips_val):>10s}{str(n_gauss):>14s}")

print("-" * len(header))
if psnr_list:
    avg_psnr = sum(psnr_list) / len(psnr_list)
    avg_ssim = sum(ssim_list) / len(ssim_list)
    avg_lpips = sum(lpips_list) / len(lpips_list)
    print(f"{'Average':<12s}{fmt(avg_psnr):>10s}{fmt(avg_ssim):>10s}{fmt(avg_lpips):>10s}{'-':>14s}")

## c07 — Packaging Submission ZIP
Zips the outputs directory into `fastgs_output_synthetic_specular.zip`.

In [ ]:
# ── Packaging Submission ZIP ──────────────────────────────────────────────────
import shutil
import os

# Robustly resolve FastGS repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/FastGS'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/FastGS'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'FastGS')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'FastGS')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'FastGS')

# Try to find the output folder to zip
candidates = [
    "/home/ghp4hc/thesis-all/fastgs_output_synthetic_specular",
    os.path.join(REPO_ROOT, "output", "synthetic_specular"),
    os.path.join(os.path.dirname(REPO_ROOT), "fastgs_output_synthetic_specular"),
]

src = None
for c in candidates:
    if os.path.isdir(c):
        src = c
        break

out = os.path.join(os.path.dirname(REPO_ROOT), "fastgs_output_synthetic_specular")

if src:
    print(f"📦 Zipping folder '{src}' -> '{out}.zip' ...")
    # Remove existing zip if exists to ensure clean write
    if os.path.exists(out + ".zip"):
        os.remove(out + ".zip")
    shutil.make_archive(out, "zip", src)
    print("archived:", out + ".zip", round(os.path.getsize(out + ".zip") / 1e6, 1), "MB")
else:
    print("❌ ERROR: Output folder not found in any of the candidate paths:", candidates)

## c08 — Hugging Face Results Upload & Auto-Cleanup
Uploads the zipped results archive back to `DiBiay/fastgs-synthetic-specular-result` and purges local zip & output files to save disk space.

In [ ]:
# ── Upload Results Zip to Hugging Face & Purge Local Files ─────────────────────
import os
import sys
import shutil
import subprocess

try:
    from huggingface_hub import HfApi
except ImportError:
    print("huggingface_hub not found. Installing via pip...")
    PROXY = 'http://rb-proxy-sl.bosch.com:8080'
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "--proxy", PROXY], check=True)
    from huggingface_hub import HfApi

# Robustly resolve FastGS repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/FastGS'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/FastGS'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'FastGS')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'FastGS')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'FastGS')

# Load Hugging Face credentials
HF_TOKEN = os.environ.get('HF_TOKEN', '<YOUR_HF_TOKEN>')
HF_REPO = os.environ.get('HF_REPO', 'DiBiay/fastgs-synthetic-specular-result')

# Try to find the zip archive file
zip_candidates = [ 
    os.path.join(os.path.dirname(REPO_ROOT), 'fastgs_output_synthetic_specular.zip'),
    os.path.join(os.getcwd(), 'fastgs_output_synthetic_specular.zip'),
    '/home/ghp4hc/thesis-all/fastgs_output_synthetic_specular.zip',
    '/home/ghp4hc/fastgs_output_synthetic_specular.zip',
]
zip_path = None
for z in zip_candidates:
    if os.path.isfile(z):
        zip_path = z
        break

if HF_TOKEN == '<YOUR_HF_TOKEN>' or not HF_TOKEN:
    print("⚠️  Hugging Face token not configured in environment variable 'HF_TOKEN'.")
    HF_TOKEN = input("Enter your Hugging Face Access Token: ").strip()

if zip_path and HF_TOKEN and HF_TOKEN != '<YOUR_HF_TOKEN>':
    print(f"🔍 Checking repository '{HF_REPO}' status...")
    api = HfApi()
    
    # Auto-create repository if it does not exist
    try:
        api.repo_info(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
        print(f"✅ Repository '{HF_REPO}' already exists.")
    except Exception:
        print(f"➕ Creating private dataset repository '{HF_REPO}'...")
        try:
            api.create_repo(
                repo_id=HF_REPO,
                repo_type="dataset",
                token=HF_TOKEN,
                private=True,
            )
            print(f"🎉 Created private repository '{HF_REPO}' successfully.")
        except Exception as e:
            print(f"⚠️  Could not create repository: {e}")

    print(f"📤 Uploading '{zip_path}' to dataset repository '{HF_REPO}'...")
    try:
        api.upload_file(
            path_or_fileobj=zip_path,
            path_in_repo="synthetic_specular.zip",
            repo_id=HF_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        print("🎉 [SUCCESS] Results archive uploaded to Hugging Face successfully!")
        
        # Purge local zip and output directory to free disk space
        out_dir = os.path.join(REPO_ROOT, "output", "synthetic_specular")
        print(f"🗑️ Cleaning up local zip file '{zip_path}' and output directory '{out_dir}'...")
        if os.path.exists(zip_path):
            os.remove(zip_path)
        if os.path.isdir(out_dir):
            shutil.rmtree(out_dir, ignore_errors=True)
        print("✅ Local disk space freed!")
    except Exception as e:
        print(f"❌ [ERROR] Hugging Face upload failed: {e}")
else:
    print(f"⚠️  Skipping upload: zip file not found or invalid HF_TOKEN.")
    if not zip_path:
        print(f"    Looked in paths: {zip_candidates}")